# ⚡ Workflows Concorrentes de Agentes com Microsoft Foundry (Python)

## 📋 Tutorial Avançado de Processamento Paralelo

Este notebook demonstra **padrões de workflows concorrentes** usando o Microsoft Agent Framework. Vai aprender a construir workflows de processamento paralelo de alto desempenho onde múltiplos agentes de IA executam simultaneamente, melhorando drasticamente a produtividade e permitindo processos empresariais sofisticados multitarefa.

> **Nota de migração:** Este exemplo fazia referência anteriormente aos Modelos do GitHub. Os Modelos do GitHub estão descontinuados (fim em Julho de 2026), por isso agora usa **Microsoft Foundry** através do `FoundryChatClient`, que aponta para a API de **Respostas** do Azure OpenAI.

## 🎯 Objetivos de Aprendizagem

### 🚀 **Fundamentos de Processamento Concorrente**
- **Execução Paralela de Agentes**: Executar múltiplos agentes simultaneamente para máxima eficiência
- **Orquestração de Workflows**: Coordenar operações concorrentes mantendo a consistência dos dados
- **Otimização de Performance**: Alcançar acelerações significativas através do processamento paralelo
- **Gestão de Recursos**: Utilizar eficientemente os recursos do modelo de IA em operações concorrentes

### 🏗️ **Padrões Avançados de Concorrência**
- **Processamento Fork-Join**: Dividir trabalho entre vários agentes e combinar resultados
- **Paralelismo de Pipeline**: Estágios de execução sobrepostos para throughput contínuo
- **Balanceamento de Carga**: Distribuir trabalho equitativamente pelos recursos dos agentes disponíveis
- **Pontos de Sincronização**: Coordenar agentes concorrentes em estágios críticos do workflow

### 🏢 **Aplicações Empresariais Concorrentes**
- **Processamento de Documentos em Alto Volume**: Processar múltiplos documentos simultaneamente
- **Análise de Conteúdo em Tempo Real**: Análise concorrente de fluxos de dados em entrada
- **Otimização de Processamento em Lote**: Maximizar o throughput para operações em grande escala
- **Análise Multimodal**: Processamento paralelo de diferentes tipos de conteúdo (texto, imagens, dados)

## ⚙️ Pré-requisitos & Configuração

### 📦 **Dependências Requeridas**

Instale o Agent Framework com capacidades para workflows concorrentes:

```bash
pip install agent-framework -U
```

### 🔑 **Configuração do Microsoft Foundry**

Inicie sessão com o Azure CLI (`az login`) para que o `AzureCliCredential` possa autenticar, depois defina os detalhes do seu projeto Microsoft Foundry.

**Configuração do Ambiente (ficheiro .env):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

**Considerações sobre Processamento Concorrente:**
- **Limites de Taxa**: Monitorizar os limites de taxa do Azure OpenAI para pedidos concorrentes
- **Uso de Recursos**: Considerar uso de memória e CPU com múltiplos agentes concorrentes
- **Tratamento de Erros**: Implementar recuperação robusta de erros para operações paralelas

### 🏗️ **Arquitetura de Workflows Concorrentes**

```mermaid
graph TD
    A[Início do Fluxo de Trabalho] --> B[Execução Concorrente]
    B --> C[Conjunto de Agentes 1]
    B --> D[Conjunto de Agentes 2]
    B --> E[Conjunto de Agentes 3]
    C --> F[Agregação de Resultados]
    D --> F
    E --> F
    F --> G[Output Final]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Principais Benefícios:**
- **⚡ Performance**: Aceleração significativa pela execução paralela
- **📈 Escalabilidade**: Lidar com cargas aumentadas sem aumento proporcional no tempo
- **🔄 Eficiência**: Melhor aproveitamento dos recursos computacionais disponíveis
- **🎯 Throughput**: Processar mais trabalho no mesmo espaço de tempo

## 🎨 **Padrões de Design de Workflows Concorrentes**

### 🔍 **Pipeline de Investigação & Análise**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Workflow de Processamento de Dados**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Pipeline de Criação de Conteúdo**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Processamento Multi-etapas**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Benefícios de Performance Empresariais**

### ⚡ **Otimização do Throughput**
- **Execução Paralela**: Múltiplos agentes a trabalhar simultaneamente
- **Utilização de Recursos**: Máxima eficácia da capacidade disponível do modelo de IA
- **Redução de Tempo**: Diminuição significativa do tempo total de processamento
- **Arquitetura Escalável**: Adicionar facilmente mais agentes concorrentes conforme necessário

### 🛡️ **Confiabilidade & Resiliência**
- **Tolerância a Falhas**: Falhas individuais de agentes não param todo o workflow
- **Isolamento de Erros**: Problemas num ramo concorrente não afetam os outros
- **Degradação Elegante**: O sistema continua a operar mesmo com capacidade reduzida de agentes
- **Mecanismos de Recuperação**: Retentativa automática e tratamento de erros para operações falhadas

### 📊 **Monitorização & Observabilidade**
- **Rastreamento de Execução Concorrente**: Monitorizar progresso de todas as operações paralelas
- **Métricas de Performance**: Medir aceleração e ganhos de eficiência
- **Análise de Uso de Recursos**: Otimizar a alocação de agentes concorrentes
- **Identificação de Gargalos**: Encontrar e resolver restrições de performance

Vamos construir workflows concorrentes de IA de alto desempenho! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
